Remove chunks that contain very long tokens. These chunks are usually the result of bad processing or undesirable content like ids, urls, etc.

In [ ]:
DB_PATH = '../backend/data.db'

SAMPLE_SIZE = 100000

In [2]:
import pandas as pd
import sqlite3
import numpy as np 

conn = sqlite3.Connection(DB_PATH)

In [6]:
df = pd.read_sql(f"""
    select chunk, chunk_id
    from chunks
    order by random()
    limit {SAMPLE_SIZE}
""", conn)

lens = []
for chunk in df.chunk:
    for token in chunk.split():
        lens.append(len(token))

lens = pd.Series(lens)
threshold = lens.mean() + lens.std() * 5
print(f"Threshold: {threshold}")

Threshold: 22.02783449069157


In [7]:
chunked_df = pd.read_sql(f"""
    select chunk, chunk_id
    from chunks
""", conn, chunksize=100000)

to_filter = []
for i, df in enumerate(chunked_df):
    print(i)
    for idx, row in df.iterrows():
        for token in row['chunk'].split():
            if len(token) > threshold:
                to_filter.append(row['chunk_id'])
print(len(to_filter))

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
810253


In [13]:
cur = conn.cursor()
cur.execute('drop table if exists to_filter')
conn.commit()

(pd
    .DataFrame(list(set(to_filter)))
    .rename(columns={0: 'chunk_id'})
    .to_sql('to_filter', conn, index=False)
)

452407

In [17]:
pd.read_sql('select count(*) from chunks', conn)

,count(*)
0,17121920


In [22]:
cur.execute("""
    delete from chunks
    where chunk_id in (
        select chunk_id
        from to_filter
    )
""")
conn.commit()

cur.execute("""
    delete from embeddings
    where chunk_id in (
        select chunk_id
        from to_filter
    )
""")
conn.commit()

In [7]:
cur.execute("""
    delete from articles
    where article_id in (
        select a.article_id
        from articles a 
        left join chunks c 
            using (article_id) 
        where c.chunk_id is null
    )
""")
conn.commit()

In [23]:
pd.read_sql('select count(*) from chunks', conn)

,count(*)
0,17121920


In [28]:
cur = conn.cursor()
cur.execute('drop table if exists to_filter')
conn.commit()

In [ ]:
# import sqlite3

# TEMP_DB_PATH = 'D:/data.db'

# conn = sqlite3.Connection(TEMP_DB_PATH)

# cur = conn.cursor()
# cur.execute('vacuum')
# conn.commit()